# RQ3: Nightmare language predicting PHQ-9 / GAD-7 by Country

Analyze USA and Argentina separately using off‑the‑shelf sentiment tools.
Predictors: VADER sentiment, TextBlob sentiment, optional embedding-based nightmare intensity.
Outcomes: `phq_total`, `gad_total`.

## 0. Setup
Install if needed:
```bash
pip install pandas scikit-learn nltk textblob sentence-transformers
```

In [52]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score, mean_absolute_error

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

try:
    from sentence_transformers import SentenceTransformer

    EMBEDDING_AVAILABLE = True
except Exception:
    SentenceTransformer = None
    EMBEDDING_AVAILABLE = False

try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    nltk.download("vader_lexicon")

pd.set_option("display.max_colwidth", 140)
DATA_PATH = Path("Upd Integration") / "10 Feb Updates" / "integrated_dream_data_v0.csv"

## 1. Common cleaning
- Drop very short or placeholder dreams
- Coerce outcomes to numeric

In [53]:
raw = pd.read_csv(DATA_PATH)
print("raw shape", raw.shape)

raw["dream_text"] = raw["dream_text"].astype(str).str.strip()
raw = raw[raw["dream_text"].str.len() >= 20].copy()
placeholders = [
    "no dreams",
    "no dream",
    "dont remember",
    "do not remember",
    "cannot remember",
    "cant remember",
    "none that i remember",
]
mask_placeholder = (
    raw["dream_text"].str.lower().apply(lambda t: any(p in t for p in placeholders))
)
raw = raw[~mask_placeholder].copy()

raw["phq_total"] = pd.to_numeric(raw.get("phq_total"), errors="coerce")
raw["gad_total"] = pd.to_numeric(raw.get("gad_total"), errors="coerce")
raw = raw.dropna(subset=["phq_total", "gad_total", "country"])
print("after filters", raw.shape)
raw[["country", "dream_text"]].head()

raw shape (9841, 24)
after filters (9684, 24)


,country,dream_text
1,USA,I don't remember dreams
2,USA,"I had a dream that my old cats were still alive. They were getting old and had cat Alzheimer's in my dream, but they were all alive and ..."
3,USA,I dreamt I encountered someone from my past. We embraced and I felt joy at reconnecting
4,USA,I had a couple nightmare dreams
6,USA,Don't remember any dreams.


## 2. Nightmare Rate
For each country subset:
- VADER: compound/pos/neg/neu
- TextBlob: polarity/subjectivity
- Nightmare intensity: Sentence-BERT similarity to fear anchors; fallback = VADER neg

In [54]:
df_us = raw[raw["country"] == "USA"].copy()

In [55]:
from empath import Empath
import numpy as np

lexicon = Empath()

nightmare_categories = [
    "fear",
    "violence",
    "death",
    "negative_emotion",
    "crime",
    "injury",
]


def empath_nightmare_rate(text: str) -> float:
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0.0
    scores = lexicon.analyze(text, categories=nightmare_categories, normalize=True)
    return float(sum(scores.values()) * 100)


df_us["nightmare_content_rate"] = df_us["dream_text"].apply(empath_nightmare_rate)

# Zero-inflation handling (important)
df_us["nightmare_any"] = (df_us["nightmare_content_rate"] > 0).astype(int)
df_us["nightmare_content_log"] = np.log1p(df_us["nightmare_content_rate"])

In [56]:
df_us[
    ["dream_text", "nightmare_content_rate", "nightmare_any", "nightmare_content_log"]
].head()

,dream_text,nightmare_content_rate,nightmare_any,nightmare_content_log
1,I don't remember dreams,0.0,0,0.0
2,"I had a dream that my old cats were still alive. They were getting old and had cat Alzheimer's in my dream, but they were all alive and ...",0.0,0,0.0
3,I dreamt I encountered someone from my past. We embraced and I felt joy at reconnecting,0.0,0,0.0
4,I had a couple nightmare dreams,0.0,0,0.0
6,Don't remember any dreams.,0.0,0,0.0


In [57]:
df_us[["nightmare_content_rate"]].describe()

,nightmare_content_rate
count,1858.000000
mean,1.893803
std,4.221289
min,0.000000
25%,0.000000
50%,0.000000
75%,2.414677
max,44.444444


## 3. Train/test split and modeling helper

In [58]:
import sys

print("Python:", sys.version)
print("Executable:", sys.executable)

try:
    import tiktoken

    print("tiktoken OK:", tiktoken.__version__)
except Exception as e:
    print("tiktoken NOT available:", repr(e))

Python: 3.12.6 | packaged by conda-forge | (main, Sep 11 2024, 04:55:15) [Clang 17.0.6 ]
Executable: /opt/miniconda3/bin/python
tiktoken OK: 0.12.0


In [59]:
import os, numpy as np
from transformers import pipeline

os.environ["TOKENIZERS_PARALLELISM"] = "false"

labels = ["fear", "threat", "danger", "violence", "distress"]

zsc = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=-1,
)

In [60]:
import numpy as np

labels = ["fear", "threat", "danger", "violence", "distress"]

texts = df_us["dream_text"].fillna("").astype(str).tolist()

outs = zsc(texts, candidate_labels=labels, multi_label=True, batch_size=16)

df_us["nightmare_intensity"] = [float(np.mean(o["scores"])) for o in outs]

In [61]:
df_us["nightmare_intensity"].describe()

count    1858.000000
mean        0.445119
std         0.278976
min         0.000740
25%         0.200709
50%         0.439488
75%         0.688634
max         0.994326
Name: nightmare_intensity, dtype: float64

In [62]:
df_us[["nightmare_content_rate", "nightmare_intensity"]].corr()

,nightmare_content_rate,nightmare_intensity
nightmare_content_rate,1.000000,0.226749
nightmare_intensity,0.226749,1.000000


In [63]:
from nltk.sentiment import SentimentIntensityAnalyzer

# Initialize VADER
sia = SentimentIntensityAnalyzer()


def get_vader_scores(text):
    if pd.isna(text) or text == "":
        return 0.0, 0.0
    scores = sia.polarity_scores(str(text))
    return scores["compound"], scores["neg"]


# Apply to df_us
df_us[["vader_compound", "vader_neg"]] = df_us["dream_text"].apply(
    lambda x: pd.Series(get_vader_scores(x))
)

print("VADER scores added to df_us.")
print(df_us[["dream_text", "vader_compound", "nightmare_intensity"]].head())

VADER scores added to df_us.
                                                                                                                                    dream_text  \
1                                                                                                                      I don't remember dreams   
2  I had a dream that my old cats were still alive. They were getting old and had cat Alzheimer's in my dream, but they were all alive and ...   
3                                                      I dreamt I encountered someone from my past. We embraced and I felt joy at reconnecting   
4                                                                                                              I had a couple nightmare dreams   
6                                                                                                                   Don't remember any dreams.   

   vader_compound  nightmare_intensity  
1         -0.3089             0.764239  
2          0

## 4. Run per country

In [64]:
analysis_df.head()

,source_file,country,language,timepoint,participant_id,dream_text,dream_feelings,dream_talk,dream_write,dream_content,...,gender,sleep_quality,sleep_hours,sleep_disturbed,nightmare_content_rate,nightmare_any,nightmare_content_log,nightmare_intensity,vader_compound,vader_neg
1,"Dream Follow Up 1_April 14, 2024_22.21.csv",USA,English,2,63386e75b02c4d7013c1198b,I don't remember dreams,NaN,No,No,No,...,Non-binary / third gender,Fairly bad,5.0,Multiple times a night,0.0,0,-0.641816,1.144209,-0.823882,3.618769
2,"Dream Follow Up 1_April 14, 2024_22.21.csv",USA,English,2,6090578fb2299eacaec32b7a,"I had a dream that my old cats were still alive. They were getting old and had cat Alzheimer's in my dream, but they were all alive and ...",Average,Yes,No,No,...,Female,Fairly good,8.0,Once a week,0.0,0,-0.641816,-1.385092,1.710873,-0.777306
3,"Dream Follow Up 1_April 14, 2024_22.21.csv",USA,English,2,58d876b14240e50001190090,I dreamt I encountered someone from my past. We embraced and I felt joy at reconnecting,Average,No,Yes,No,...,Male,Fairly bad,6.0,2-3 nights a week,0.0,0,-0.641816,-1.587886,1.007593,-0.777306
4,"Dream Follow Up 1_April 14, 2024_22.21.csv",USA,English,2,655a395e2828d333bae0ea4b,I had a couple nightmare dreams,Average,No,No,No,...,Male,Fairly good,8.0,Once a week,0.0,0,-0.641816,1.338064,0.630982,-0.777306
6,"Dream Follow Up 1_April 14, 2024_22.21.csv",USA,English,2,659daa1ed4e13d6428103c1f,Don't remember any dreams.,NaN,No,No,Yes,...,Female,Fairly good,7.0,Once a night,0.0,0,-0.641816,1.595457,-0.823882,2.781026


In [65]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# 1. Prepare data and drop NaNs
# Note: Ensure these columns exist in df_us
predictors = ["nightmare_intensity", "nightmare_content_log", "vader_compound"]
df_analysis = df_us.dropna(subset=predictors + ["phq_total", "gad_total"]).copy()

# 2. Scale predictors (Crucial for comparing coefficients)
scaler = StandardScaler()
df_analysis[predictors] = scaler.fit_transform(df_analysis[predictors])


def run_stable_rq3(target):
    print(f"\n{'='*20} Results for {target.upper()} {'='*20}")

    # Add a constant (intercept) - OLS needs this explicitly in statsmodels
    X = sm.add_constant(df_analysis[predictors])
    y = df_analysis[target]

    # Fit OLS model with Robust Standard Errors ('HC3' is standard for small/medium samples)
    model = sm.OLS(y, X).fit(cov_type="HC3")

    print(model.summary())
    return model


# Run for both
phq_res = run_stable_rq3("phq_total")
gad_res = run_stable_rq3("gad_total")


==================== Results for PHQ_TOTAL ====================
                            OLS Regression Results                            
Dep. Variable:              phq_total   R-squared:                       0.016
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     9.677
Date:                Wed, 11 Feb 2026   Prob (F-statistic):           2.45e-06
Time:                        02:09:12   Log-Likelihood:                -5900.5
No. Observations:                1858   AIC:                         1.181e+04
Df Residuals:                    1854   BIC:                         1.183e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------

In [66]:
df_analysis["phq_total"].describe()

count    1858.000000
mean        5.949946
std         5.840933
min         0.000000
25%         1.000000
50%         4.000000
75%         9.000000
max        27.000000
Name: phq_total, dtype: float64

## 5. Quick visuals (optional, USA shown as example)

In [67]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

usa = raw_all[raw_all["country"].str.lower() == "usa"]
if not usa.empty:
    usa = add_features(usa)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.scatterplot(ax=axes[0], data=usa, x="vader_compound", y="phq_total", alpha=0.4)
    axes[0].set_title("USA: VADER compound vs PHQ")
    sns.scatterplot(ax=axes[1], data=usa, x="vader_compound", y="gad_total", alpha=0.4)
    axes[1].set_title("USA: VADER compound vs GAD")
    plt.tight_layout()
    plt.show()

NameError: name 'add_features' is not defined

## 6. Next steps
- Swap in RoBERTa/DistilBERT sentiment for richer context.
- Add emotion labels (GoEmotions) or LIWC-derived affect terms.
- Run mixed-effects to account for repeated measures and country interactions.